In [1]:
import os
from pathlib import Path

print("Searching for all video files...\n")

video_locations = {}
for root, dirs, files in os.walk("/kaggle/input"):
    mp4s = [f for f in files if f.endswith('.mp4')]
    if mp4s:
        folder = Path(root)
        # Identify by filename patterns
        if any("_kling" in f for f in mp4s):
            video_locations["kling"] = folder
        elif any("_seedance" in f for f in mp4s):
            video_locations["seedance"] = folder
        elif any("_gemini" in f for f in mp4s):
            video_locations["gemini_omni_flash"] = folder
        elif any(f.startswith("pexels_") for f in mp4s):
            video_locations["pexels"] = folder
        print(f"  {folder.name}: {len(mp4s)} mp4 files")
        for f in sorted(mp4s)[:2]:
            print(f"    sample: {f}")
        print()

print("Identified sources:")
for name, path in video_locations.items():
    count = len(list(path.glob("*.mp4")))
    print(f"  {name}: {count} videos at {path}")

# Find prompts file
prompts_path = None
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if "week2_prompts" in f and f.endswith(".txt"):
            prompts_path = Path(root) / f
            break

print(f"\nPrompts file: {prompts_path}")

Searching for all video files...

  gemini_omni_flash: 32 mp4 files
    sample: w2_001_gemini_omni_flash.mp4
    sample: w2_002_gemini_omni_flash.mp4

  kling: 32 mp4 files
    sample: w2_001_kling.mp4
    sample: w2_002_kling.mp4

  seedance: 32 mp4 files
    sample: w2_001_seedance.mp4
    sample: w2_002_seedance.mp4

Identified sources:
  gemini_omni_flash: 32 videos at /kaggle/input/datasets/shantanuvedanteog/commercial-renamed/commercial/gemini_omni_flash
  kling: 32 videos at /kaggle/input/datasets/shantanuvedanteog/commercial-renamed/commercial/kling
  seedance: 32 videos at /kaggle/input/datasets/shantanuvedanteog/commercial-renamed/commercial/seedance

Prompts file: /kaggle/input/datasets/shantanuvedanteog/week2-prompts-v1generation-txt/week2_prompts_v1(generation).txt


In [2]:
import json
import re
import subprocess
from collections import Counter

with open(prompts_path) as f:
    prompt_lines = [line.strip() for line in f 
                    if line.strip() and not line.strip().startswith("#") and "|" in line]

prompts_lookup = {}
for line in prompt_lines:
    pid, category, prompt_text = line.split("|", 2)
    prompts_lookup[pid] = {
        "category": category,
        "prompt_text": prompt_text,
    }

CINEMATIC_SUFFIX = ", natural lighting, cinematic quality, high detail, realistic"
print(f"Loaded {len(prompts_lookup)} prompts")

Loaded 40 prompts


In [3]:
NORMALISED_ROOT = Path("/kaggle/working/normalised")

gemini_out = NORMALISED_ROOT / "gemini_omni_flash"
gemini_out.mkdir(parents=True, exist_ok=True)

gemini_src = video_locations["gemini_omni_flash"]
gemini_videos = sorted(gemini_src.glob("*.mp4"))
print(f"Normalising {len(gemini_videos)} Gemini Omni Flash videos...")
print("  Stripping audio + trimming to 3 sec + 832x480 + 24fps\n")

failed = []
for i, src in enumerate(gemini_videos):
    dst = gemini_out / src.name
    cmd = [
        "ffmpeg", "-y", "-i", str(src),
        "-t", "3",
        "-vf", "scale=832:480:force_original_aspect_ratio=decrease,"
               "pad=832:480:(ow-iw)/2:(oh-ih)/2:color=black",
        "-r", "24",
        "-c:v", "libx264",
        "-preset", "slow",
        "-crf", "18",
        "-pix_fmt", "yuv420p",
        "-an",
        str(dst)
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        failed.append((src.name, result.stderr[-150:]))
    if (i+1) % 10 == 0:
        print(f"  Done {i+1}/{len(gemini_videos)}")

print(f"\nGemini: {len(gemini_videos) - len(failed)} success, {len(failed)} failed")
if failed:
    for name, err in failed[:3]:
        print(f"  FAILED: {name}: {err}")

Normalising 32 Gemini Omni Flash videos...
  Stripping audio + trimming to 3 sec + 832x480 + 24fps

  Done 10/32
  Done 20/32
  Done 30/32

Gemini: 32 success, 0 failed


In [4]:
seedance_out = NORMALISED_ROOT / "seedance"
seedance_out.mkdir(parents=True, exist_ok=True)

seedance_src = video_locations["seedance"]
seedance_videos = sorted(seedance_src.glob("*.mp4"))
print(f"Normalising {len(seedance_videos)} Seedance 2.0 videos...")
print("  Trimming to 3 sec + 832x480 + 24fps\n")

failed = []
for i, src in enumerate(seedance_videos):
    dst = seedance_out / src.name
    cmd = [
        "ffmpeg", "-y", "-i", str(src),
        "-t", "3",
        "-vf", "scale=832:480:force_original_aspect_ratio=decrease,"
               "pad=832:480:(ow-iw)/2:(oh-ih)/2:color=black",
        "-r", "24",
        "-c:v", "libx264",
        "-preset", "slow",
        "-crf", "18",
        "-pix_fmt", "yuv420p",
        "-an",
        str(dst)
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        failed.append((src.name, result.stderr[-150:]))
    if (i+1) % 10 == 0:
        print(f"  Done {i+1}/{len(seedance_videos)}")

print(f"\nSeedance: {len(seedance_videos) - len(failed)} success, {len(failed)} failed")

Normalising 32 Seedance 2.0 videos...
  Trimming to 3 sec + 832x480 + 24fps

  Done 10/32
  Done 20/32
  Done 30/32

Seedance: 32 success, 0 failed


In [5]:
kling_out = NORMALISED_ROOT / "kling"
kling_out.mkdir(parents=True, exist_ok=True)

kling_src = video_locations["kling"]
kling_videos = sorted(kling_src.glob("*.mp4"))
print(f"Normalising {len(kling_videos)} Kling 3.0 videos...")
print("  Rescaling to 832x480 + 24fps (already 3 sec, no audio)\n")

failed = []
for i, src in enumerate(kling_videos):
    dst = kling_out / src.name
    cmd = [
        "ffmpeg", "-y", "-i", str(src),
        "-t", "3",
        "-vf", "scale=832:480:force_original_aspect_ratio=decrease,"
               "pad=832:480:(ow-iw)/2:(oh-ih)/2:color=black",
        "-r", "24",
        "-c:v", "libx264",
        "-preset", "slow",
        "-crf", "18",
        "-pix_fmt", "yuv420p",
        "-an",
        str(dst)
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        failed.append((src.name, result.stderr[-150:]))
    if (i+1) % 10 == 0:
        print(f"  Done {i+1}/{len(kling_videos)}")

print(f"\nKling: {len(kling_videos) - len(failed)} success, {len(failed)} failed")

Normalising 32 Kling 3.0 videos...
  Rescaling to 832x480 + 24fps (already 3 sec, no audio)

  Done 10/32
  Done 20/32
  Done 30/32

Kling: 32 success, 0 failed


In [11]:
def get_video_info(video_path):
    cmd = [
        "ffprobe", "-v", "error",
        "-show_entries", "stream=width,height,r_frame_rate,duration,codec_name",
        "-show_entries", "format=size,duration,bit_rate",
        "-of", "json",
        str(video_path)
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0:
        try:
            data = json.loads(result.stdout)
            stream = data.get("streams", [{}])[0]
            fmt = data.get("format", {})
            fps_parts = stream.get("r_frame_rate", "0/1").split("/")
            fps = round(int(fps_parts[0]) / int(fps_parts[1]), 2) if len(fps_parts) == 2 and int(fps_parts[1]) > 0 else 0
            return {
                "width": stream.get("width"),
                "height": stream.get("height"),
                "fps": fps,
                "duration_sec": round(float(stream.get("duration", fmt.get("duration", 0))), 2),
                "codec": stream.get("codec_name"),
                "filesize_bytes": int(fmt.get("size", 0)),
                "bitrate_bps": int(fmt.get("bit_rate", 0)),
            }
        except:
            pass
    return {}


# Normalisation specs (same for all four after processing)
NORMALISATION = {
    "target_width": 832,
    "target_height": 480,
    "target_fps": 24,
    "target_duration_sec": 3,
    "codec": "libx264",
    "crf": 18,
    "preset": "slow",
    "audio": "stripped",
    "aspect_handling": "letterbox with black padding",
}

# Source-specific info
source_info = {
    "kling": {
        "generator": "Kling 3.0",
        "source_type": "commercial",
        "access_method": "Higgsfield trial",
        "generation_date": "2026-07-21",
        "license": "Generated via Higgsfield trial; Kling 3.0 by Kuaishou",
        "native_resolution": "1280x720",
        "native_duration_sec": 3,
        "native_audio": False,
    },
    "gemini_omni_flash": {
        "generator": "Gemini Omni Flash",
        "source_type": "commercial",
        "access_method": "Higgsfield trial",
        "generation_date": "2026-07-21",
        "license": "Generated via Higgsfield trial; Gemini Omni Flash by Google",
        "native_resolution": "1280x720",
        "native_duration_sec": 4,
        "native_audio": True,
        "normalisation_note": "Audio stripped, trimmed from 4 sec to 3 sec for cross-generator consistency",
    },
    "seedance": {
        "generator": "Seedance 2.0",
        "source_type": "commercial",
        "access_method": "Higgsfield trial",
        "generation_date": "2026-07-22",
        "license": "Generated via Higgsfield trial; Seedance 2.0 by ByteDance",
        "native_resolution": "1280x720",
        "native_duration_sec": 4,
        "native_audio": False,
        "normalisation_note": "Trimmed from 4 sec to 3 sec for cross-generator consistency",
    },
}


METADATA_ROOT = Path("/kaggle/working/metadata")

for source_key, info in source_info.items():
    source_dir = NORMALISED_ROOT / source_key
    meta_per_video = METADATA_ROOT / source_key
    meta_per_video.mkdir(parents=True, exist_ok=True)
    
    videos = sorted(source_dir.glob("*.mp4"))
    print(f"\n=== {info['generator']} ({len(videos)} videos) ===")
    
    all_metadata = []
    
    for v in videos:
        # Get actual specs after normalisation
        specs = get_video_info(v)
        
        # Base metadata
        meta = {
            "video_id": v.name,
            "generator": info["generator"],
            "source_type": info["source_type"],
            "access_method": info["access_method"],
            "license": info["license"],
            "normalisation": NORMALISATION,
            "actual_specs": specs,
        }
        
        # Add prompt info for commercial models
        if source_key != "pexels":
            pid_match = re.match(r'(w2_\d+)', v.stem)
            if pid_match:
                pid = pid_match.group(1)
                prompt_info = prompts_lookup.get(pid, {})
                meta["prompt_id"] = pid
                meta["category"] = prompt_info.get("category", "unknown")
                meta["original_prompt"] = prompt_info.get("prompt_text", "")
                meta["full_prompt"] = prompt_info.get("prompt_text", "") + CINEMATIC_SUFFIX
            meta["generation_date"] = info["generation_date"]
            meta["native_specs"] = {
                "resolution": info["native_resolution"],
                "duration_sec": info["native_duration_sec"],
                "audio": info["native_audio"],
            }
            if "normalisation_note" in info:
                meta["normalisation_note"] = info["normalisation_note"]
        
        all_metadata.append(meta)
        
        # Save per-video JSON
        with open(meta_per_video / f"{v.stem}.json", "w") as f:
            json.dump(meta, f, indent=2)
    
    # Save combined JSON
    combined_path = METADATA_ROOT / f"{source_key}_metadata.json"
    with open(combined_path, "w") as f:
        json.dump(all_metadata, f, indent=2)
    
    # Category breakdown
    categories = Counter(m.get("category", "unknown") for m in all_metadata)
    for cat, count in sorted(categories.items()):
        print(f"  {cat}: {count}")
    print(f"  Total: {len(all_metadata)}")


=== Kling 3.0 (32 videos) ===
  animal: 4
  edge_case: 4
  hands: 4
  motion: 4
  multi_person: 4
  portrait: 4
  text_scene: 4
  texture: 4
  Total: 32

=== Gemini Omni Flash (32 videos) ===
  animal: 4
  edge_case: 4
  hands: 4
  motion: 4
  multi_person: 4
  portrait: 4
  text_scene: 4
  texture: 4
  Total: 32

=== Seedance 2.0 (32 videos) ===
  animal: 4
  edge_case: 4
  hands: 4
  motion: 4
  multi_person: 4
  portrait: 4
  text_scene: 4
  texture: 4
  Total: 32


In [12]:
print("=" * 60)
print("FINAL CORPUS SUMMARY")
print("=" * 60)

grand_total = 0
for source in ["kling", "gemini_omni_flash", "seedance"]:
    norm_dir = NORMALISED_ROOT / source
    meta_combined = METADATA_ROOT / f"{source}_metadata.json"
    
    video_count = len(list(norm_dir.glob("*.mp4")))
    
    # Verify one video's specs
    sample = list(norm_dir.glob("*.mp4"))[0]
    specs = get_video_info(sample)
    
    print(f"\n{source}:")
    print(f"  Videos: {video_count}")
    print(f"  Sample specs: {specs.get('width')}x{specs.get('height')}, "
          f"{specs.get('fps')} fps, {specs.get('duration_sec')} sec, "
          f"{specs.get('bitrate_bps', 0)//1000} kbps")
    grand_total += video_count

print(f"\n{'='*60}")
print(f"GRAND TOTAL: {grand_total} normalised videos with metadata")
print(f"All at 832x480, 24fps, 3 sec, CRF 18, no audio")

FINAL CORPUS SUMMARY

kling:
  Videos: 32
  Sample specs: 832x480, 24.0 fps, 3.0 sec, 1819 kbps

gemini_omni_flash:
  Videos: 32
  Sample specs: 832x480, 24.0 fps, 3.0 sec, 1576 kbps

seedance:
  Videos: 32
  Sample specs: 832x480, 24.0 fps, 3.0 sec, 1522 kbps

GRAND TOTAL: 96 normalised videos with metadata
All at 832x480, 24fps, 3 sec, CRF 18, no audio


In [13]:
import shutil
shutil.make_archive("/kaggle/working/normalised", "zip")
!ls -la /kaggle/working/*.zip 
print("\nReady to download gemini_omni_flash.zip")
print("from the Output panel on the right sidebar.")

-rw-r--r-- 1 root root 57615158 Jul 23 14:23 /kaggle/working/normalised.zip

Ready to download gemini_omni_flash.zip
from the Output panel on the right sidebar.


In [14]:
import shutil
shutil.make_archive("/kaggle/working/metadata", "zip")
!ls -la /kaggle/working/*.zip 
print("\nReady to download metadata.zip")
print("from the Output panel on the right sidebar.")

-rw-r--r-- 1 root root 115223766 Jul 23 14:24 /kaggle/working/metadata.zip
-rw-r--r-- 1 root root  57615158 Jul 23 14:23 /kaggle/working/normalised.zip

Ready to download metadata.zip
from the Output panel on the right sidebar.
